# Block 4 — KG rescoring

Trusted ticks only (`is_marked` and not HiTL). Write-ins via `assume()`. Observed tubes from Block 3 digits — **never** copy expected into empty crops.

Do **not** upload clinic PHI.


## 0. Clone the live tree


In [ ]:
# Colab does not clone src/ when you open a GitHub notebook. Pull the live tree.
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/RwaRwa599/epq3.git"
BRANCH = "block1"

def _run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.check_call(cmd)

def ensure_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, Path("/content/epq3")]:
        if (cand / "src" / "med_doc").is_dir() and (cand / "pyproject.toml").exists():
            os.chdir(cand)
            return cand
    dest = Path("/content/epq3") if Path("/content").is_dir() else (here / "epq3")
    if not (dest / ".git").is_dir():
        url = REPO
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        if token:
            url = f"https://{token}@github.com/RwaRwa599/epq3.git"
        _run(["git", "clone", "--branch", BRANCH, "--single-branch", url, str(dest)])
    else:
        _run(["git", "-C", str(dest), "fetch", "origin", BRANCH])
        _run(["git", "-C", str(dest), "checkout", BRANCH])
        _run(["git", "-C", str(dest), "pull", "--ff-only", "origin", BRANCH])
    os.chdir(dest)
    return dest

root = ensure_repo()
_run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "matplotlib"])
print("cwd:", os.getcwd())
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet

In [ ]:
from med_doc.htr.batch import process_from_block1
from med_doc.kg import KnowledgeGraph
from med_doc.normalization.batch import normalize_batch
from med_doc.rescoring import process_from_block3
from med_doc.review import ReviewPatch, process_from_block4

kg = KnowledgeGraph.load()

def ensure_block1():
    z = OUT / "block1.zip"
    if z.exists():
        return z
    return Path(normalize_batch([demo_sheet()], output_dir=OUT / "b1", output_zip=z)["output_zip"])

def ensure_block3():
    z = OUT / "block3.zip"
    if z.exists():
        return z
    b1 = ensure_block1()
    return Path(process_from_block1(b1, output_dir=OUT / "b3", output_zip=z, kg=kg, backend="lexicon", mode="both")["output_zip"])

def ensure_block4():
    z = OUT / "block4.zip"
    if z.exists():
        return z
    b3 = ensure_block3()
    return Path(process_from_block3(b3, output_dir=OUT / "b4", output_zip=z, kg=kg)["output_zip"])

## 1. Rescore Block 3 drafts


In [ ]:
b4 = process_from_block3(ensure_block3(), output_dir=OUT / "b4", output_zip=OUT / "block4.zip", kg=kg)
doc = b4["manifest"]["documents"][0]
print("ticked", doc["ticked_test_ids"])
print("expected", doc["expected_tubes"])
print("observed", doc["observed_tubes"])
print("hitl", doc["hitl_fields"])
print("discrepancies", doc["discrepancies"])
pred = json.loads((OUT / "b4" / "docs" / doc["doc_id"] / "prediction.json").read_text())
hyp = json.loads((OUT / "b4" / "docs" / doc["doc_id"] / "hypotheses.json").read_text())
print("hypotheses tube source", hyp["verbal"].get("tube_edta", {}).get("source"))
assert hyp["verbal"].get("tube_edta", {}).get("source") != "prior_expected"
print("prediction tube canonical", pred["handwriting_fields"].get("tube_edta", {}).get("canonical_value"))
download(OUT / "block4.zip")